# Ultimate IDPS DNN Training Pipeline (AWS S3 Version)
**Description:**
This notebook trains the deep learning model by streaming data directly from a public AWS S3 bucket instead of downloading massive files locally.

**Steps:**
1. Ensure the `S3_URL` in Cell 3 points to your actual dataset.
2. Run all cells in order.
3. Download `idps_model.keras` + `idps_scaler.pkl` to your IDPS root directory.

In [ ]:
# CELL 1: Install Dependencies
!pip install -q s3fs dask tensorflow scikit-learn pandas numpy matplotlib seaborn

In [ ]:
# CELL 2: Define Feature Schema & Loader Functions
import pandas as pd
import numpy as np
import gc
import warnings
warnings.filterwarnings("ignore")

CANONICAL_FEATURES = [
    "Destination Port", "Flow Duration",
    "Total Fwd Packets", "Total Backward Packets",
    "Total Length of Fwd Packets", "Total Length of Bwd Packets",
    "Fwd Packet Length Max", "Fwd Packet Length Min",
    "Fwd Packet Length Mean", "Fwd Packet Length Std",
    "Bwd Packet Length Max", "Bwd Packet Length Min",
    "Bwd Packet Length Mean", "Bwd Packet Length Std",
    "Flow Bytes/s", "Flow Packets/s",
    "Flow IAT Mean", "Flow IAT Std", "Flow IAT Max", "Flow IAT Min",
    "Fwd IAT Total", "Fwd IAT Mean", "Fwd IAT Std", "Fwd IAT Max", "Fwd IAT Min",
    "Bwd IAT Total", "Bwd IAT Mean", "Bwd IAT Std", "Bwd IAT Max", "Bwd IAT Min",
    "Fwd PSH Flags", "Bwd PSH Flags", "Fwd URG Flags", "Bwd URG Flags",
    "Fwd Header Length", "Bwd Header Length",
    "Fwd Packets/s", "Bwd Packets/s",
    "Min Packet Length", "Max Packet Length",
    "Packet Length Mean", "Packet Length Std", "Packet Length Variance",
    "FIN Flag Count", "SYN Flag Count", "RST Flag Count",
    "PSH Flag Count", "ACK Flag Count", "URG Flag Count",
    "CWE Flag Count", "ECE Flag Count",
    "Down/Up Ratio", "Average Packet Size",
    "Avg Fwd Segment Size", "Avg Bwd Segment Size",
    "Fwd Header Length.1",
    "Fwd Avg Bytes/Bulk", "Fwd Avg Packets/Bulk", "Fwd Avg Bulk Rate",
    "Bwd Avg Bytes/Bulk", "Bwd Avg Packets/Bulk", "Bwd Avg Bulk Rate",
    "Subflow Fwd Packets", "Subflow Fwd Bytes",
    "Subflow Bwd Packets", "Subflow Bwd Bytes",
    "Init_Win_bytes_forward", "Init_Win_bytes_backward",
    "act_data_pkt_fwd", "min_seg_size_forward",
    "Active Mean", "Active Std", "Active Max", "Active Min",
    "Idle Mean", "Idle Std", "Idle Max", "Idle Min",
]
print(f"Canonical features: {len(CANONICAL_FEATURES)}")

def strip_cols(df):
    df.columns = [str(c).strip() for c in df.columns]
    return df

def find_label_col(df):
    for c in ["Label", "label", "Attack", "attack", "attack_cat",
              "Attack_type", "type", "class", "Class"]:
        if c in df.columns:
            return c
    for c in df.columns:
        if "label" in c.lower() or "attack" in c.lower() or "class" in c.lower():
            return c
    return None

def binarize_label(series):
    s = series.astype(str).str.strip().str.upper()
    benign = ["BENIGN", "NORMAL", "0", "LEGITIMATE", "NAN"]
    return (~s.isin(benign)).astype(int)

print("Functions ready.")

In [ ]:
# CELL 3: Load Data Directly From Public S3 Bucket
# === REPLACE WITH YOUR ACTUAL S3 URL ===
    S3_URL = "s3://cse-cic-ids2018/Processed Traffic Data for ML Algorithms/Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv"

print(f"Streaming data from {S3_URL} ...")
all_frames = []

try:
    # We read in chunks to avoid memory overflow.
    # We'll sample from each chunk to build a representative dataset.
    chunk_iterator = pd.read_csv(
        S3_URL, 
        chunksize=100000, 
        storage_options={"anon": True}, 
        low_memory=False, 
        on_bad_lines="skip"
    )
    
    for i, chunk in enumerate(chunk_iterator):
        chunk = strip_cols(chunk)
        label_col = find_label_col(chunk)
        if label_col is None:
            continue
            
        y = binarize_label(chunk[label_col])
        matched = [f for f in CANONICAL_FEATURES if f in chunk.columns]
        
        X = chunk[matched].copy()
        for feat in CANONICAL_FEATURES:
            if feat not in X.columns:
                X[feat] = 0.0
        X = X[CANONICAL_FEATURES]
        X = X.apply(pd.to_numeric, errors="coerce").fillna(0)
        X.replace([np.inf, -np.inf], 0, inplace=True)
        
        X["__label__"] = y.values
        
        # Keep a fraction of the chunk to keep RAM usage manageable (e.g. 20%)
        sampled = X.sample(frac=0.2, random_state=42)
        all_frames.append(sampled)
        
        if i >= 49: # Stop after 5 million rows processed (50 chunks) to save time/memory, adjust as needed.
            break
            
    combined = pd.concat(all_frames, ignore_index=True)
    del all_frames
    gc.collect()
    print(f"Successfully loaded and sampled {len(combined):,} rows.")
    print(f"Attack ratio: {combined['__label__'].mean():.2%}")
    
except Exception as e:
    print(f"Error reading from S3: {e}")
    print("Make sure the S3 URL is correct and public.")

In [ ]:
# CELL 4: Balance Dataset & Train/Test Split
from sklearn.model_selection import train_test_split

if 'combined' in locals():
    X = combined[CANONICAL_FEATURES].values.astype(np.float64)
    y = combined["__label__"].values.astype(np.int32)
    del combined
    gc.collect()

    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    benign_idx = np.where(y == 0)[0]
    attack_idx = np.where(y == 1)[0]
    print(f"Before: BENIGN={len(benign_idx):,}, ATTACK={len(attack_idx):,}")

    max_majority = len(attack_idx) * 2
    if len(benign_idx) > max_majority:
        np.random.seed(42)
        benign_idx = np.random.choice(benign_idx, max_majority, replace=False)

    keep_idx = np.concatenate([benign_idx, attack_idx])
    np.random.shuffle(keep_idx)
    X = X[keep_idx]
    y = y[keep_idx]
    print(f"After:  BENIGN={np.sum(y==0):,}, ATTACK={np.sum(y==1):,}")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y)
    print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")

In [ ]:
# CELL 5: Scale Features
from sklearn.preprocessing import StandardScaler
import pickle

if 'X_train' in locals():
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    with open("/content/idps_scaler.pkl", "wb") as f:
        pickle.dump(scaler, f)
    print("Scaler saved to /content/idps_scaler.pkl")

In [ ]:
# CELL 6: Build & Train DNN
import tensorflow as tf
from tensorflow.keras import layers, callbacks

if 'X_train' in locals():
    tf.random.set_seed(42)

    model = tf.keras.Sequential([
        layers.Input(shape=(78,)),
        layers.Dense(512, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        layers.Dense(256, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(128, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.1),
        layers.Dense(2, activation="softmax"),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"])

    model.summary()

    MODEL_PATH = "/content/idps_model.keras"

    cbs = [
        callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
        callbacks.ModelCheckpoint(MODEL_PATH, monitor="val_accuracy", save_best_only=True, verbose=1),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
    ]

    history = model.fit(
        X_train, y_train,
        validation_split=0.15,
        epochs=50,
        batch_size=2048,
        callbacks=cbs,
        verbose=1)

    print(f"\nModel saved to {MODEL_PATH}")

In [ ]:
# CELL 7: Evaluate
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

if 'model' in locals():
    model = tf.keras.models.load_model(MODEL_PATH)
    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

    print(classification_report(y_test, y_pred, target_names=["BENIGN", "ATTACK"]))

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["BENIGN", "ATTACK"], yticklabels=["BENIGN", "ATTACK"])
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()

    loss, acc = model.evaluate(X_test, y_test, verbose=0)
    print(f"\nTest Accuracy: {acc:.4f}")

In [ ]:
# CELL 8: Download Model Files
from google.colab import files
import os

if os.path.exists("/content/idps_model.keras") and os.path.exists("/content/idps_scaler.pkl"):
    print("Downloading model files...")
    files.download("/content/idps_model.keras")
    files.download("/content/idps_scaler.pkl")
    print("\nPlace both files in your IDPS project root directory.")
else:
    print("Model files not found. Ensure training completed successfully.")